# 01 — Performance: Mixed Precision, Gradient Accumulation, Clipping, Profiling

**Overview:** FP32/FP16/BF16 memory and range comparison, mixed precision via `torch.autocast`,
gradient accumulation verified equivalent to a larger batch, gradient clipping taming an
exploding update, and simple data-loading-vs-compute profiling.

**Required libraries:** `torch`.

**Note:** this machine has no CUDA GPU (CPU and, if available, Apple's MPS backend only).
`torch.autocast` supports CPU with `bfloat16`, which is what this notebook uses — the same
autocast API works with `device_type="cuda"` and FP16 on a CUDA GPU, with `GradScaler` added
for FP16's narrower numeric range (not needed for BF16).

Connects to: `22_pytorch_training` (the pipeline being optimized) and `17_deep_learning_architectures`
(gradient clipping relates directly to the vanishing/exploding gradient discussion there).
Closes Day 3's PyTorch sequence before `24_huggingface_foundations`.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **FP32** | 32-bit floating point | Single precision; default training dtype. |
| **FP16** | 16-bit floating point | Half precision; half the memory but narrow range (needs loss scaling). |
| **BF16** | Brain Floating Point 16 | 16-bit format with FP32's exponent range but less precision; no loss scaling needed. |
| **CUDA** | Compute Unified Device Architecture | NVIDIA's GPU programming platform; PyTorch's `cuda` device. |
| **GPU** | Graphics Processing Unit | Massively parallel processor; the main accelerator for deep learning. |
| **CPU** | Central Processing Unit | General-purpose processor; few powerful cores. |
| **MPS** | Metal Performance Shaders | Apple's GPU backend for PyTorch on Apple-silicon Macs. |
| **API** | Application Programming Interface | The set of functions/classes a library exposes for you to call. |
| **ReLU** | Rectified Linear Unit | Activation max(0, x). |
| **MSE** | Mean Squared Error | Average of squared prediction errors; the standard regression loss. |

## 0. Environment

In [1]:
import torch
import torch.nn as nn
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("mps available:", torch.backends.mps.is_available())
torch.manual_seed(42)


torch: 2.14.1
cuda available: False
mps available: True


## 1. FP32 vs FP16 vs BF16: memory and numeric range

💡 **Intuition:** FP16 and BF16 both use 16 bits (half of FP32's 32), but split those bits
differently between exponent (numeric RANGE) and mantissa (numeric PRECISION).

In [2]:
x_fp32 = torch.randn(1000, 1000, dtype=torch.float32)
x_fp16 = x_fp32.half()      # torch.float16
x_bf16 = x_fp32.bfloat16()  # torch.bfloat16

bytes_per_element = {"fp32": x_fp32.element_size(), "fp16": x_fp16.element_size(), "bf16": x_bf16.element_size()}
print("bytes per element:", bytes_per_element)
total_elements = x_fp32.numel()
for name, size in bytes_per_element.items():
    print(f"{name}: {total_elements * size / 1024:.1f} KB for this 1000x1000 tensor")

assert bytes_per_element["fp32"] == 4
assert bytes_per_element["fp16"] == bytes_per_element["bf16"] == 2

# Numeric range: BF16 keeps FP32's exponent range; FP16 has a much narrower range.
print("\nFP16 max representable value: ~65504")
print("BF16 max representable value: ~3.4e38 (same order as FP32)")
large_value = torch.tensor([70000.0])
print("\n70000.0 cast to fp16:", large_value.half().item(), "(overflowed to 'inf' -- exceeds FP16's range)")
print("70000.0 cast to bf16:", large_value.bfloat16().item(), "(representable, though less precisely than FP32)")

assert torch.isinf(large_value.half()).item()
assert not torch.isinf(large_value.bfloat16()).item()


bytes per element: {'fp32': 4, 'fp16': 2, 'bf16': 2}
fp32: 3906.2 KB for this 1000x1000 tensor
fp16: 1953.1 KB for this 1000x1000 tensor
bf16: 1953.1 KB for this 1000x1000 tensor

FP16 max representable value: ~65504
BF16 max representable value: ~3.4e38 (same order as FP32)

70000.0 cast to fp16: inf (overflowed to 'inf' -- exceeds FP16's range)
70000.0 cast to bf16: 70144.0 (representable, though less precisely than FP32)


## 2. Mixed precision with `torch.autocast`

💡 **Intuition:** `autocast` automatically runs eligible operations (mostly matmuls) in a
lower-precision dtype for speed/memory, while keeping numerically sensitive operations in
FP32 -- you don't have to manually cast every tensor.

In [3]:
model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))
x = torch.randn(32, 64)

with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    output = model(x)
    print("output dtype inside autocast:", output.dtype)

output_outside = model(x)
print("output dtype outside autocast:", output_outside.dtype)

assert output.dtype == torch.bfloat16
assert output_outside.dtype == torch.float32
print("\nautocast changed the computation's dtype without any manual .half()/.bfloat16() calls on the model or input.")


output dtype inside autocast: torch.bfloat16
output dtype outside autocast: torch.float32

autocast changed the computation's dtype without any manual .half()/.bfloat16() calls on the model or input.


## 3. Gradient accumulation: simulating a larger batch

**Equation:** accumulating gradients over `k` micro-batches of size `B` (without stepping the
optimizer between them) approximates one batch of size `k*B`, since both average the same
per-sample gradients.

In [4]:
torch.manual_seed(0)
model_a = nn.Linear(10, 1)
model_b = nn.Linear(10, 1)
model_b.load_state_dict(model_a.state_dict())   # identical starting weights

X_full = torch.randn(16, 10)
y_full = torch.randn(16, 1)
loss_fn = nn.MSELoss()

# Version A: one big batch of 16.
model_a.zero_grad()
loss_a = loss_fn(model_a(X_full), y_full)
loss_a.backward()
grad_a = model_a.weight.grad.clone()

# Version B: 4 micro-batches of 4, accumulated (gradients summed, then averaged by dividing by num micro-batches).
model_b.zero_grad()
num_micro_batches = 4
micro_batch_size = 4
for i in range(num_micro_batches):
    X_micro = X_full[i * micro_batch_size:(i + 1) * micro_batch_size]
    y_micro = y_full[i * micro_batch_size:(i + 1) * micro_batch_size]
    loss_b = loss_fn(model_b(X_micro), y_micro) / num_micro_batches   # scale down before accumulating
    loss_b.backward()   # gradients ACCUMULATE across this loop (no zero_grad between micro-batches)
grad_b = model_b.weight.grad.clone()

print("gradient from one batch of 16:        ", grad_a)
print("gradient from 4 accumulated batches of 4:", grad_b)
print("max abs difference:", (grad_a - grad_b).abs().max().item())

assert torch.allclose(grad_a, grad_b, atol=1e-5)
print("\ngradient accumulation over micro-batches matches one larger batch, as expected for MSE.")


gradient from one batch of 16:         tensor([[-0.4071, -0.4851, -0.9197, -0.2403, -0.1280,  0.1107, -1.1245, -0.2436,
         -0.6218,  0.4770]])
gradient from 4 accumulated batches of 4: tensor([[-0.4071, -0.4851, -0.9197, -0.2403, -0.1280,  0.1107, -1.1245, -0.2436,
         -0.6218,  0.4770]])
max abs difference: 1.4901161193847656e-07

gradient accumulation over micro-batches matches one larger batch, as expected for MSE.


## 4. Gradient clipping: taming an exploding update

⚠️ **Common Mistake:** no safety net against an occasional bad batch producing a huge
gradient -- a single destabilizing update can undo many steps of good progress.

In [5]:
model_c = nn.Linear(10, 1)
X_bad = torch.randn(4, 10) * 100       # an artificially extreme batch
y_bad = torch.randn(4, 1) * 100

loss_bad = loss_fn(model_c(X_bad), y_bad)
loss_bad.backward()

grad_norm_before = torch.nn.utils.clip_grad_norm_(model_c.parameters(), max_norm=float("inf"))  # just measures, doesn't clip
print(f"gradient norm from the extreme batch (unclipped): {grad_norm_before:.2f}")

model_c.zero_grad()
loss_bad2 = loss_fn(model_c(X_bad), y_bad)
loss_bad2.backward()
clipped_norm = torch.nn.utils.clip_grad_norm_(model_c.parameters(), max_norm=1.0)
actual_norm_after_clipping = torch.sqrt(sum((p.grad ** 2).sum() for p in model_c.parameters()))

print(f"gradient norm reported by clip_grad_norm_ (the PRE-clip norm): {clipped_norm:.2f}")
print(f"actual gradient norm AFTER clipping: {actual_norm_after_clipping:.4f} (should be ~1.0, the max_norm)")

assert actual_norm_after_clipping <= 1.0 + 1e-3


gradient norm from the extreme batch (unclipped): 15249.77
gradient norm reported by clip_grad_norm_ (the PRE-clip norm): 15249.77
actual gradient norm AFTER clipping: 1.0000 (should be ~1.0, the max_norm)


## 5. Profiling: is the bottleneck data loading or compute?

💡 **Intuition:** before optimizing model compute, confirm that's actually where the time
goes -- a slow `Dataset.__getitem__` (recap from `21_pytorch_data`) can dominate total time
even with a fast model.

In [6]:
import time
from torch.utils.data import Dataset, DataLoader

class ArtificiallySlowDataset(Dataset):
    def __init__(self, n, slow=True):
        self.n = n
        self.slow = slow
    def __len__(self):
        return self.n
    def __getitem__(self, idx):
        if self.slow:
            time.sleep(0.005)   # simulate expensive preprocessing
        return torch.randn(10), torch.randint(0, 2, (1,)).squeeze()

def profile_one_epoch(dataset, model):
    loader = DataLoader(dataset, batch_size=8)
    data_time, compute_time = 0.0, 0.0
    t_prev = time.perf_counter()
    for X_batch, y_batch in loader:
        t_data_end = time.perf_counter()
        data_time += t_data_end - t_prev

        logits = model(X_batch)
        loss = nn.functional.cross_entropy(logits, y_batch)
        loss.backward()

        t_prev = time.perf_counter()
        compute_time += t_prev - t_data_end
    return data_time, compute_time

tiny_model = nn.Linear(10, 2)
slow_data_time, slow_compute_time = profile_one_epoch(ArtificiallySlowDataset(80, slow=True), tiny_model)
fast_data_time, fast_compute_time = profile_one_epoch(ArtificiallySlowDataset(80, slow=False), tiny_model)

print(f"SLOW dataset -- data loading: {slow_data_time:.3f}s, compute: {slow_compute_time:.3f}s "
      f"(data-bound: {slow_data_time / (slow_data_time + slow_compute_time):.0%} of total time)")
print(f"FAST dataset -- data loading: {fast_data_time:.3f}s, compute: {fast_compute_time:.3f}s "
      f"(data-bound: {fast_data_time / (fast_data_time + fast_compute_time):.0%} of total time)")

print("\nWith the slow dataset, most time goes to data loading -- no amount of model/GPU "
      "optimization would help until the Dataset itself (or num_workers, from 21_pytorch_data) "
      "is fixed.")
assert slow_data_time > fast_data_time


SLOW dataset -- data loading: 0.484s, compute: 0.008s (data-bound: 98% of total time)
FAST dataset -- data loading: 0.002s, compute: 0.004s (data-bound: 29% of total time)

With the slow dataset, most time goes to data loading -- no amount of model/GPU optimization would help until the Dataset itself (or num_workers, from 21_pytorch_data) is fixed.


## 6. Connection to what's next

These techniques — mixed precision, gradient accumulation, clipping, and profiling-before-
optimizing — are used throughout this workspace's `llm-fine-tuning/` curriculum, where model
and batch sizes make naive full-precision, full-batch, unprofiled training impractical.
`24_huggingface_foundations` closes this curriculum's Day 3 by introducing the Hugging Face
stack these techniques are usually applied through in practice.

## 7. Interview questions
1. Why does BF16 overflow less easily than FP16, despite both being 16-bit?
2. What specifically does `torch.autocast` do, and what stays in FP32 even inside an autocast context?
3. Why does gradient accumulation need the loss divided by the number of micro-batches before `.backward()`?
4. Why does `clip_grad_norm_` return the pre-clipping norm rather than the post-clipping one?
5. How would you determine whether a slow training loop is data-bound or compute-bound?

## 8. Exercises
1. Repeat the gradient accumulation experiment with a loss function that's NOT a simple average (e.g. a sum) and observe how the equivalence breaks.
2. Sweep `clip_grad_norm_`'s `max_norm` from 0.1 to 10 and observe how aggressively each value constrains an exploding gradient.
3. Extend the profiling experiment to also measure time spent in the optimizer step, and report the full breakdown (data / forward+backward / optimizer).

## 9. Summary

✅ BF16 trades FP32's precision for half the memory while keeping its exponent range; mixed
precision automates using lower precision where safe; gradient accumulation simulates a larger
batch; gradient clipping caps destabilizing updates; profiling identifies the REAL bottleneck
before optimizing. This closes `23_pytorch_performance` — next: `24_huggingface_foundations`.